<a href="https://colab.research.google.com/github/bsneha90/llm_engineering/blob/main/RealTimeAudioTranscribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio transformers torch torchaudio numpy scipy

In [ ]:
pip show gradio


Name: gradio
Version: 5.50.0
Summary: Python library for easily interacting with trained machine learning models
Home-page: https://github.com/gradio-app/gradio
Author: 
Author-email: Abubakar Abid <gradio-team@huggingface.co>, Ali Abid <gradio-team@huggingface.co>, Ali Abdalla <gradio-team@huggingface.co>, Dawood Khan <gradio-team@huggingface.co>, Ahsen Khaliq <gradio-team@huggingface.co>, Pete Allen <gradio-team@huggingface.co>, Ömer Faruk Özdemir <gradio-team@huggingface.co>, Freddy A Boulton <gradio-team@huggingface.co>, Hannah Blair <gradio-team@huggingface.co>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiofiles, anyio, brotli, fastapi, ffmpy, gradio-client, groovy, httpx, huggingface-hub, jinja2, markupsafe, numpy, orjson, packaging, pandas, pillow, pydantic, pydub, python-multipart, pyyaml, ruff, safehttpx, semantic-version, starlette, tomlkit, typer, typing-extensions, uvicorn
Required-by: 


In [ ]:
"""
Meeting Transcriber — Real-time Whisper + Gradio
"""

import gradio as gr
import torch
import numpy as np
import datetime
import wave
import io
from pathlib import Path
import soundfile as sf


from transformers import pipeline

# ── Model ──────────────────────────────────────────────────────────────────
# Options: "openai/whisper-tiny"  (fastest, least accurate)
#          "openai/whisper-base"
#          "openai/whisper-small"
#          "openai/whisper-medium"
#          "openai/whisper-large-v3" (slowest, most accurate)
MODEL_ID = "openai/whisper-base.en"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Whisper model '{MODEL_ID}' on {device}…")

transcriber = pipeline(
    "automatic-speech-recognition",
    model=MODEL_ID,
    device=device,
    chunk_length_s=30,
    stride_length_s=5,
    return_timestamps=False,   # timestamps slow streaming; disabled for speed
)






Loading Whisper model 'openai/whisper-base.en' on cuda…


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [ ]:
# # ── Shared state ───────────────────────────────────────────────────────────
# # Using lists so Gradio's per-session state can hold mutable references.
# # Replace with gr.State() objects if you need multi-user isolation.
# full_transcript: list[dict] = []   # [{"time": "HH:MM:SS", "text": "…"}, …]
# audio_chunks: list[np.ndarray] = []  # raw float32 arrays
# audio_sample_rate: list[int] = [16000]

# # Minimum audio duration (seconds) to bother transcribing a streaming chunk.
# # Lower = more responsive; too low causes Whisper to hallucinate.
# MIN_CHUNK_SECS = 1.5


# # ── Core logic ─────────────────────────────────────────────────────────────

# def _transcribe_array(sr: int, audio: np.ndarray) -> str:
#     """Run Whisper on a numpy audio array, return stripped text."""
#     if audio.ndim > 1:
#         audio = audio.mean(axis=1)           # stereo → mono
#     audio = audio.astype("float32")
#     if audio.dtype.kind == "i":              # int16 → float32
#         audio = audio / 32768.0
#     duration = len(audio) / sr
#     if duration < MIN_CHUNK_SECS:
#         return ""
#     result = transcriber({"sampling_rate": sr, "raw": audio})
#     return result["text"].strip()


# def stream_transcribe(audio, transcript_md):
#     """Called every ~1-2 s while the microphone is live (streaming=True)."""
#     if audio is None:
#         return transcript_md

#     sr, chunk = audio
#     audio_sample_rate[0] = sr

#     # Save chunk for audio export
#     audio_chunks.append(chunk.copy())

#     text = _transcribe_array(sr, chunk)
#     if text:
#         ts = datetime.datetime.now().strftime("%H:%M:%S")
#         full_transcript.append({"time": ts, "text": text})

#     return _format_transcript(full_transcript)


# def _format_transcript(entries: list[dict]) -> str:
#     if not entries:
#         return "*No transcript yet — start speaking…*"
#     return "\n\n".join(f"**[{e['time']}]** {e['text']}" for e in entries)



MIN_CHUNK_SECS = 0.5  # example

audio_chunks: list[np.ndarray] = []
audio_sample_rate = [None]
full_transcript: list[dict] = []


def _normalize_audio(audio: np.ndarray) -> np.ndarray:
    """Convert to mono float32 in range [-1, 1]."""
    if audio.ndim > 1:
        audio = audio.mean(axis=1)  # stereo → mono

    # Convert int → float32 BEFORE casting
    if np.issubdtype(audio.dtype, np.integer):
        max_val = np.iinfo(audio.dtype).max
        audio = audio.astype(np.float32) / max_val
    else:
        audio = audio.astype(np.float32)

    return audio


def _transcribe_array(sr: int, audio: np.ndarray) -> str:
    """Run Whisper on a numpy audio array, return stripped text."""
    audio = _normalize_audio(audio)

    duration = len(audio) / sr
    if duration < MIN_CHUNK_SECS:
        return ""

    result = transcriber({"sampling_rate": sr, "raw": audio})
    return result["text"].strip()


def stream_transcribe(audio, transcript_md):
    """Called every ~1–2 s while microphone is live (streaming=True)."""
    if audio is None:
        return transcript_md

    sr, chunk = audio

    # Ensure sample rate consistency
    if audio_sample_rate[0] is None:
        audio_sample_rate[0] = sr
    elif audio_sample_rate[0] != sr:
        raise ValueError("Sample rate changed during streaming!")

    # Normalize once and store consistently
    normalized_chunk = _normalize_audio(chunk)
    audio_chunks.append(normalized_chunk.copy())

    text = _transcribe_array(sr, chunk)
    if text:
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        full_transcript.append({"time": ts, "text": text})

    return _format_transcript(full_transcript)


def save_full_audio(filepath="full_recording.wav"):
    """Merge saved chunks and export to WAV."""
    if not audio_chunks:
        return None

    full_audio = np.concatenate(audio_chunks, axis=0)
    sf.write(filepath, full_audio, audio_sample_rate[0])
    return filepath


def _format_transcript(entries: list[dict]) -> str:
    if not entries:
        return "*No transcript yet — start speaking…*"
    return "\n\n".join(
        f"**[{e['time']}]** {e['text']}" for e in entries
    )

def clear_all():
    full_transcript.clear()
    audio_chunks.clear()
    return "*Transcript cleared. Ready for a new session.*"


# # ── Export helpers ─────────────────────────────────────────────────────────

# def export_transcript():
#     if not full_transcript:
#         return None
#     lines = ["MEETING TRANSCRIPT", "=" * 50,
#              f"Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}",
#              "=" * 50, ""]
#     for e in full_transcript:
#         lines.append(f"[{e['time']}] {e['text']}\n")
#     path = "/tmp/meeting_transcript.txt"
#     with open(path, "w") as f:
#         f.write("\n".join(lines))
#     return path


# def export_audio():
#     if not audio_chunks:
#         return None
#     sr = audio_sample_rate[0]
#     combined = np.concatenate(audio_chunks, axis=0)
#     # Normalise and convert to int16 for WAV
#     if combined.dtype.kind == "f":
#         combined = np.clip(combined, -1.0, 1.0)
#         combined = (combined * 32767).astype(np.int16)
#     path = "/tmp/meeting_audio.wav"
#     with wave.open(path, "w") as wf:
#         wf.setnchannels(1 if combined.ndim == 1 else combined.shape[1])
#         wf.setsampwidth(2)   # 16-bit
#         wf.setframerate(sr)
#         wf.writeframes(combined.tobytes())
#     return path

In [ ]:
def export_audio(filepath: str = "recording.wav") -> str:
    """Merge saved chunks and export full WAV."""
    if not audio_chunks:
        raise ValueError("No audio chunks recorded.")

    if audio_sample_rate[0] is None:
        raise ValueError("Sample rate unknown.")

    full_audio = np.concatenate(audio_chunks, axis=0)
    sf.write(filepath, full_audio, audio_sample_rate[0])
    return filepath


def export_transcript_md(filepath: str = "transcript.md") -> str:
    """Export transcript as Markdown."""
    if not full_transcript:
        raise ValueError("No transcript available.")

    content = _format_transcript(full_transcript)
    Path(filepath).write_text(content, encoding="utf-8")
    return filepath


def export_transcript_txt(filepath: str = "transcript.txt") -> str:
    """Export transcript as plain text."""
    if not full_transcript:
        raise ValueError("No transcript available.")

    lines = [
        f"[{e['time']}] {e['text']}"
        for e in full_transcript
    ]
    Path(filepath).write_text("\n".join(lines), encoding="utf-8")
    return filepath

In [ ]:
css = """
body { font-family: 'Georgia', serif; background: #0f1117; color: #e8e0d0; }
#title { text-align: center; padding: 2rem 0 0.5rem; }
#title h1 { font-size: 2.2rem; font-weight: 400; letter-spacing: 0.08em;
            color: #c9b99a; text-transform: uppercase; margin: 0; }
#title p  { color: #7a7060; font-style: italic; font-size: 0.95rem; margin: 0.3rem 0 0; }
.transcript-wrap textarea,
.transcript-wrap .prose {
    background: #181c24 !important; border: 1px solid #2e2a22 !important;
    border-radius: 8px; padding: 1.2rem; min-height: 340px;
    font-family: 'Courier New', monospace; font-size: 0.9rem;
    line-height: 1.8; color: #d4c9b8;
}
footer { visibility: hidden; }
"""

with gr.Blocks(css=css, title="Meeting Transcriber") as demo:

    with gr.Column(elem_id="title"):
        gr.HTML("<h1>🎙 Meeting Transcriber</h1>")
        gr.HTML(
            "<p>Real-time Whisper · speak naturally · transcript updates every ~2 s</p>"
        )

    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                sources=["microphone"],
                type="numpy",
                streaming=True,
                # stream_every=0.2,
                label="🎤 Press ▶ to allow mic access & start recording",
            )

            with gr.Row():
                clear_btn = gr.Button("🗑 Clear")
                txt_btn = gr.Button("📄 Export .txt")
                wav_btn = gr.Button("🔊 Export .wav")

            txt_file = gr.File(label="Transcript download")
            wav_file = gr.File(label="Audio download")

        with gr.Column(scale=2, elem_classes=["transcript-wrap"]):
            transcript_box = gr.Markdown(
                value="*No transcript yet — start speaking…*"
            )

    # Stream transcription
    audio_input.stream(
        fn=stream_transcribe,
        inputs=audio_input,
        outputs=transcript_box,
    )

    # Buttons
    clear_btn.click(clear_all, outputs=transcript_box)
    txt_btn.click(export_transcript_txt, outputs=txt_file)
    wav_btn.click(export_audio, outputs=wav_file)


if __name__ == "__main__":
    demo.launch(share=False)  # Must run on localhost or HTTPS for mic

/tmp/ipykernel_182/3677060541.py:17: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="Meeting Transcriber") as demo:
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1052: UserWarning: Expected 2 arguments for function <function stream_transcribe at 0x7d926d37fce0>, received 1.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1056: UserWarning: Expected at least 2 arguments for function <function stream_transcribe at 0x7d926d37fce0>, received 1.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>